# IEEE-CIS Credit Card Fraud Detection

**Workflow:** exploratory data analysis → chronological holdout → LightGBM model comparison → Kaggle test predictions.  
**Data:** [IEEE-CIS Fraud Detection on Kaggle](https://www.kaggle.com/competitions/ieee-fraud-detection/data).  
**Target:** `isFraud` (1 = fraudulent, 0 = legitimate).

Place Kaggle's four CSV files in the repository's `data/` directory, install `requirements.txt`, and run the notebook from its first cell. Outputs, including predictions, reports, and the trained model, are saved under `outputs/` and excluded from Git.

The 26-feature baseline LightGBM model was selected on the chronological validation set using F1 at a 0.20 decision threshold. Kaggle evaluates the **ranking probabilities using ROC-AUC**, so the F1 decision threshold is not used for the submission file. The actual Kaggle after-deadline submission returned public ROC-AUC **0.896908** and private ROC-AUC **0.870183**.

This public version removes large saved cell outputs and replaces machine-specific paths. It has not been rerun here because the Kaggle CSV files were not part of the upload.


## 1. Setup and resource checks
Run this cell first (and again after restarting the kernel). All later cells use the same `con` and `source` variables.

**Important:** Select the **Python (Fraud Project)** kernel. The expected Python executable is inside `.../envs/fraud/`. No package-install commands belong inside this analysis notebook.

In [ ]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import SVG, display

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = DATA_DIR / "train_transaction.csv"
IDENTITY_CSV = DATA_DIR / "train_identity.csv"

if not TRAIN_CSV.is_file():
    raise FileNotFoundError(f"Training dataset not found: {TRAIN_CSV}")

train_file = TRAIN_CSV.as_posix()
source = f"read_csv_auto('{train_file}')"

con = duckdb.connect(database=":memory:")
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")

print("Python:", sys.executable)
print("NumPy:", np.__version__, "| Pandas:", pd.__version__, "| DuckDB:", duckdb.__version__)
print("Training file:", TRAIN_CSV.name, f"({TRAIN_CSV.stat().st_size / 1024**2:,.1f} MiB)")
print("Identity file available:", IDENTITY_CSV.is_file())
print("Data is processed by DuckDB; only small summary tables go into Pandas.")

## 2. Dataset overview and key data-quality checks
The source data is intentionally kept in its original state. Each check below is calculated without creating a Pandas DataFrame of all transactions. Missing email domains do **not** mean the associated transactions are illegitimate.

In [ ]:
overview = con.execute(f"""
    SELECT
        COUNT(*) AS total_transactions,
        SUM(isFraud)::BIGINT AS fraud_cases,
        SUM(CASE WHEN isFraud = 0 THEN 1 ELSE 0 END)::BIGINT AS legitimate_cases,
        ROUND(100.0 * AVG(isFraud), 2) AS fraud_percentage,
        COUNT(TransactionID) - COUNT(DISTINCT TransactionID) AS duplicate_transaction_ids,
        COUNT(*) - COUNT(TransactionID) AS missing_transaction_ids,
        COUNT(*) - COUNT(isFraud) AS missing_labels,
        COUNT(*) - COUNT(TransactionAmt) AS missing_amounts,
        COUNT(*) - COUNT(ProductCD) AS missing_product_codes,
        COUNT(*) - COUNT(card1) AS missing_card1,
        COUNT(*) - COUNT(P_emaildomain) AS missing_email_domains
    FROM {source}
""").df()

display(overview.T.rename(columns={0: "Value"}))
n_transactions = int(overview.loc[0, "total_transactions"])
print("\nInterpretation: Overall fraud rate is a class prevalence, not model accuracy.")

## 3. Data dictionary and missingness across all columns
`train_transaction.csv` contains **394 columns**: one transaction identifier (`TransactionID`), one target (`isFraud`), and **392 potential predictors**. The identity file is separate and will be joined in a later phase.

We count missing values per column, not rows with any missing value. A transaction may contribute to the missing counts of multiple features.

In [ ]:
schema = con.execute(f"DESCRIBE SELECT * FROM {source}").df()
print("Total columns:", len(schema))
display(schema[["column_name", "column_type"]].head(15))

# One pass over all 394 columns. No full transaction DataFrame in memory.
missing_expressions = [
    f'COUNT(*) - COUNT("{col}") AS "{col}"'
    for col in schema["column_name"]
]
missing_sql = f"SELECT {', '.join(missing_expressions)} FROM {source}"

missing = con.execute(missing_sql).df().T.reset_index()
missing.columns = ["feature", "missing_count"]
missing["missing_count"] = missing["missing_count"].astype("int64")
missing["missing_percentage"] = 100.0 * missing["missing_count"] / n_transactions

display(
    missing.sort_values("missing_percentage", ascending=False)
    .head(20)
    .assign(missing_percentage=lambda x: x["missing_percentage"].round(2))
)

### 3.1 Missingness groups
These groups describe data availability. **They are not automatic rules for deleting features.** Even an almost-empty feature might encode useful information in whether it is present.

In [ ]:
def missing_group(pct):
    if pct == 0:
        return "Complete (0%)"
    if pct <= 50:
        return "Low/moderate (>0-50%)"
    if pct <= 90:
        return "High (>50-90%)"
    return "Extremely high (>90%)"

missing["missing_group"] = missing["missing_percentage"].apply(missing_group)
order = ["Complete (0%)", "Low/moderate (>0-50%)", "High (>50-90%)", "Extremely high (>90%)"]
missing_summary = missing["missing_group"].value_counts().reindex(order, fill_value=0)
display(missing_summary.rename_axis("Group").reset_index(name="Columns"))

print("\nFeatures exceeding 90% missingness:")
display(
    missing.loc[missing["missing_percentage"] > 90, ["feature", "missing_count", "missing_percentage"]]
    .assign(missing_percentage=lambda x: x["missing_percentage"].round(2))
)

## 4. Does feature availability relate to fraud?
We compare fraud prevalence where `dist2` and `D7` are missing versus present. In the original analysis, these features had more than 93% missingness, yet their presence was associated with higher fraud rates. **Association does not establish causation or predictive performance.**

In [ ]:
availability_tables = []

for feature in ("dist2", "D7"):
    small = con.execute(f"""
        SELECT
            '{feature}' AS feature,
            CASE WHEN "{feature}" IS NULL THEN 'Missing' ELSE 'Present' END AS availability,
            COUNT(*) AS transactions,
            SUM(isFraud)::BIGINT AS fraud_cases,
            ROUND(100.0 * AVG(isFraud), 2) AS fraud_rate_pct
        FROM {source}
        GROUP BY availability
    """).df()
    availability_tables.append(small)

availability = pd.concat(availability_tables, ignore_index=True)
display(availability.sort_values(["feature", "availability"]))

print("\nInterpretation: A missing D7 or dist2 value does not mean the transaction is legitimate.")

## 5. Transaction amount: descriptive statistics
Before testing hypotheses or training models, compare the average, median and tails of transaction amounts. `TransactionAmt` is an amount feature; the competition does not guarantee that it is denominated in a single currency.

In [ ]:
amount_summary = con.execute(f"""
    SELECT
        CASE WHEN isFraud = 1 THEN 'Fraud' ELSE 'Legitimate' END AS transaction_type,
        COUNT(*) AS transactions,
        ROUND(AVG(TransactionAmt), 2) AS mean_amount,
        ROUND(QUANTILE_CONT(TransactionAmt, 0.05), 2) AS p05,
        ROUND(QUANTILE_CONT(TransactionAmt, 0.25), 2) AS q1,
        ROUND(MEDIAN(TransactionAmt), 2) AS median_amount,
        ROUND(QUANTILE_CONT(TransactionAmt, 0.75), 2) AS q3,
        ROUND(QUANTILE_CONT(TransactionAmt, 0.95), 2) AS p95,
        ROUND(MIN(TransactionAmt), 2) AS minimum_amount,
        ROUND(MAX(TransactionAmt), 2) AS maximum_amount
    FROM {source}
    GROUP BY isFraud
    ORDER BY isFraud
""").df()

display(amount_summary[[
    "transaction_type", "transactions", "mean_amount", "median_amount",
    "minimum_amount", "maximum_amount", "p95"
]])
print("\nBoth mean and median are useful; the mean is sensitive to large values.")

## 6. Low-memory visualizations (no Matplotlib)
The uploaded notebook's last Matplotlib test showed printed version information but no completed chart, and Jupyter reported kernel restarts. The exact cause cannot be established from that file alone. To avoid both GUI/backend crashes and unnecessary memory use, the graphs below use **DuckDB aggregates + inline SVG**. They do not import Matplotlib or transfer all 590,540 rows into Pandas.

The histogram is restricted to an upper limit derived from the groups' 95th-percentile amounts. Each group is separately normalized **within the displayed amount range**; its bar height is *not* a fraud probability.

In [ ]:
# Compute just 2 x 30 histogram counts (rather than loading all amounts).
N_BINS = 30
amount_limit = float(amount_summary["p95"].max())

hist = con.execute(f"""
    SELECT
        isFraud,
        LEAST({N_BINS - 1}, CAST(FLOOR(TransactionAmt / {amount_limit} * {N_BINS}) AS INTEGER)) AS amount_bin,
        COUNT(*) AS transactions
    FROM {source}
    WHERE TransactionAmt >= 0 AND TransactionAmt <= {amount_limit}
    GROUP BY isFraud, amount_bin
    ORDER BY isFraud, amount_bin
""").df()

# Fill empty bins; normalize within the displayed amount range for each class.
counts = (
    hist.pivot(index="amount_bin", columns="isFraud", values="transactions")
    .reindex(index=range(N_BINS), columns=[0, 1], fill_value=0)
    .fillna(0)
)
shares = counts.div(counts.sum(axis=0), axis=1) * 100

W, H = 960, 430
LEFT, RIGHT, TOP, BOTTOM = 90, 28, 77, 77
PW, PH = W - LEFT - RIGHT, H - TOP - BOTTOM
bar_group = PW / N_BINS
max_y = float(shares.to_numpy().max()) * 1.14
max_y = max(max_y, 1)
colors = {0: "#3976A8", 1: "#E19A42"}
parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="{W}" height="{H}">',
         '<rect width="100%" height="100%" fill="white"/>',
         '<text x="90" y="34" font-size="21" font-family="Arial" font-weight="bold">Transaction amount distribution</text>',
         f'<text x="90" y="54" font-size="13" fill="#596575" font-family="Arial">Normalized within 0-{amount_limit:,.0f}; extreme amounts excluded from this view</text>']

for t in range(5):
    fraction = t / 4
    y = TOP + PH * (1 - fraction)
    parts += [f'<line x1="{LEFT}" y1="{y:.1f}" x2="{LEFT+PW}" y2="{y:.1f}" stroke="#e1e5ec"/>',
              f'<text x="{LEFT-12}" y="{y+4:.1f}" text-anchor="end" font-family="Arial" font-size="12" fill="#536070">{max_y*fraction:.1f}%</text>']

for i in range(N_BINS):
    for pos, label in enumerate((0, 1)):
        share = float(shares.loc[i, label])
        height = PH * share / max_y
        x = LEFT + i * bar_group + bar_group * (0.10 + pos * 0.42)
        y = TOP + PH - height
        parts.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{bar_group*0.38:.2f}" height="{height:.2f}" fill="{colors[label]}"/>')

for i in range(0, N_BINS + 1, 5):
    x = LEFT + PW * i / N_BINS
    parts.append(f'<text x="{x:.2f}" y="{TOP+PH+24}" text-anchor="middle" font-family="Arial" font-size="12" fill="#536070">{amount_limit*i/N_BINS:,.0f}</text>')

parts += [f'<line x1="{LEFT}" y1="{TOP+PH}" x2="{LEFT+PW}" y2="{TOP+PH}" stroke="#65707c"/>',
          f'<text x="{LEFT+PW/2}" y="{H-20}" text-anchor="middle" font-size="14" font-family="Arial">Transaction amount</text>',
          f'<rect x="{W-286}" y="19" width="13" height="13" fill="{colors[0]}"/>',
          f'<text x="{W-267}" y="30" font-size="13" font-family="Arial">Legitimate</text>',
          f'<rect x="{W-167}" y="19" width="13" height="13" fill="{colors[1]}"/>',
          f'<text x="{W-148}" y="30" font-size="13" font-family="Arial">Fraud</text>',
          '</svg>']

display(SVG(data="".join(parts)))
print("This graph compares distribution shapes, not the probability of fraud at each amount.")

### 6.1 Percentile box summary
This box-style chart shows **5th-to-95th-percentile whiskers**, **25th-to-75th-percentile boxes**, and median lines. This is an explicitly defined percentile summary, not a conventional 1.5-IQR Tukey boxplot. The full data are retained.

In [ ]:
# Render a simple box-style graph using the aggregates already calculated.
W, H = 960, 330
LEFT, RIGHT = 145, 36
PW = W - LEFT - RIGHT
scale_max = float(amount_summary["p95"].max()) * 1.08
scale_x = lambda v: LEFT + float(v) / scale_max * PW

parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="{W}" height="{H}">',
         '<rect width="100%" height="100%" fill="white"/>',
         '<text x="45" y="38" font-size="21" font-weight="bold" font-family="Arial">Transaction amount: percentile box comparison</text>',
         '<text x="45" y="60" font-size="13" fill="#596575" font-family="Arial">Whiskers = 5th-95th percentiles; box = Q1-Q3; line = median</text>']

for i in range(6):
    val = (scale_max / 5) * i
    x = scale_x(val)
    parts += [f'<line x1="{x:.2f}" y1="88" x2="{x:.2f}" y2="270" stroke="#e1e5ec"/>',
              f'<text x="{x:.2f}" y="292" text-anchor="middle" font-family="Arial" font-size="12" fill="#536070">{val:,.0f}</text>']

for name, y, color in (("Legitimate", 136, "#3976A8"), ("Fraud", 224, "#E19A42")):
    row = amount_summary.loc[amount_summary["transaction_type"] == name].iloc[0]
    p05, q1, med, q3, p95 = (scale_x(row[k]) for k in ("p05", "q1", "median_amount", "q3", "p95"))
    parts += [f'<text x="{LEFT-18}" y="{y+5}" text-anchor="end" font-family="Arial" font-size="15" fill="#273541">{name}</text>',
              f'<line x1="{p05:.1f}" y1="{y}" x2="{p95:.1f}" y2="{y}" stroke="{color}" stroke-width="3"/>',
              f'<line x1="{p05:.1f}" y1="{y-10}" x2="{p05:.1f}" y2="{y+10}" stroke="{color}" stroke-width="2"/>',
              f'<line x1="{p95:.1f}" y1="{y-10}" x2="{p95:.1f}" y2="{y+10}" stroke="{color}" stroke-width="2"/>',
              f'<rect x="{q1:.1f}" y="{y-19}" width="{max(1,q3-q1):.1f}" height="38" fill="{color}" fill-opacity="0.28" stroke="{color}" stroke-width="2"/>',
              f'<line x1="{med:.1f}" y1="{y-19}" x2="{med:.1f}" y2="{y+19}" stroke="{color}" stroke-width="3"/>']

parts += [f'<text x="{LEFT+PW/2}" y="{H-10}" text-anchor="middle" font-family="Arial" font-size="14">Transaction amount</text>', '</svg>']
display(SVG(data="".join(parts)))

## 7. Findings and interpretation
- **Data completeness:** The original analysis identified 52 fully observed columns, 168 with >0-50% missingness, 172 with >50-90%, and two (`dist2`, `D7`) with >90%. Recompute above to verify against the source.
- **Fraud prevalence:** 20,663 of 590,540 transactions (3.50%) were labeled fraudulent. Class imbalance means a model's accuracy alone is inadequate.
- **High-missingness features:** `dist2` and `D7` may be useful because their *availability* differs substantially between fraud and legitimate records. Retain for now; test on future held-out transactions.
- **Transaction amount:** The observed mean was 134.51 (legitimate) versus 149.24 (fraud); median 68.50 versus 75.00. These are descriptive associations, not causes or demonstrated model performance.
- **Evaluation rule:** No hypothesis-test p-value or feature should be treated as proof of useful prediction without a chronologically held-out evaluation.

## 8. Completed modelling workflow
The subsequent cells merge identity data, define a chronological holdout, compare three LightGBM configurations, retrain the selected model, score Kaggle test transactions, and export evaluation reports. The reported validation figures were produced before retraining on all labelled data.


In [ ]:

# Step 12: Fraud rate by transaction amount range

amount_risk = con.execute(f"""
    WITH amount_groups AS (
        SELECT
            isFraud,
            CASE
                WHEN TransactionAmt < 25 THEN '0-25'
                WHEN TransactionAmt < 50 THEN '25-50'
                WHEN TransactionAmt < 100 THEN '50-100'
                WHEN TransactionAmt < 200 THEN '100-200'
                WHEN TransactionAmt < 500 THEN '200-500'
                WHEN TransactionAmt < 1000 THEN '500-1000'
                ELSE '1000+'
            END AS amount_range,

            CASE
                WHEN TransactionAmt < 25 THEN 1
                WHEN TransactionAmt < 50 THEN 2
                WHEN TransactionAmt < 100 THEN 3
                WHEN TransactionAmt < 200 THEN 4
                WHEN TransactionAmt < 500 THEN 5
                WHEN TransactionAmt < 1000 THEN 6
                ELSE 7
            END AS sort_order

        FROM {source}
        WHERE TransactionAmt IS NOT NULL
          AND TransactionAmt >= 0
    )

    SELECT
        amount_range,
        COUNT(*) AS total_transactions,
        SUM(isFraud)::BIGINT AS fraud_cases,
        ROUND(100.0 * AVG(isFraud), 2) AS fraud_rate_pct

    FROM amount_groups
    GROUP BY amount_range, sort_order
    ORDER BY sort_order
""").df()

display(amount_risk)


In [ ]:

from IPython.display import HTML, display

# Visualize fraud rate across all amount ranges
max_rate = amount_risk["fraud_rate_pct"].max()

bars = ""

for _, row in amount_risk.iterrows():

    width = row["fraud_rate_pct"] / max_rate * 100

    bars += f"""
    <div style="display:flex; align-items:center;
                gap:12px; margin:12px 0;">

        <div style="width:90px; text-align:right;">
            {row['amount_range']}
        </div>

        <div style="width:350px; background:#e5e7eb;
                    height:24px;">
            <div style="width:{width:.1f}%;
                        background:#287D91;
                        height:100%;">
            </div>
        </div>

        <div>
            <b>{row['fraud_rate_pct']:.2f}%</b>
        </div>

    </div>
    """

display(HTML(
    "<h3>Fraud Rate by Transaction Amount Range</h3>"
    + bars
))


In [ ]:

import numpy as np
from scipy.stats import chi2_contingency

# Use the amount_risk summary already calculated
fraud = amount_risk["fraud_cases"].to_numpy()

legitimate = (
    amount_risk["total_transactions"]
    - amount_risk["fraud_cases"]
).to_numpy()

# Create the contingency table
contingency_table = np.column_stack(
    [legitimate, fraud]
)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

# Cramer's V: strength of association
n = contingency_table.sum()
cramers_v = np.sqrt(
    chi2 / (n * min(contingency_table.shape[0] - 1,
                    contingency_table.shape[1] - 1))
)

print("Chi-Square Statistic:", round(chi2, 4))
print("Degrees of Freedom:", dof)
print("P-value:", "< 1e-300" if p_value == 0 else f"{p_value:.4e}")
print("Cramer's V:", round(cramers_v, 4))

if p_value < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


In [ ]:

import duckdb
from IPython.display import SVG, display

# 1. Load product-category fraud statistics
csv_path = str(TRAIN_CSV)

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

product_risk = con.execute("""
    SELECT
        ProductCD,
        COUNT(*) AS total_transactions,
        SUM(isFraud)::BIGINT AS fraud_cases,
        ROUND(100.0 * AVG(isFraud), 2) AS fraud_rate_pct
    FROM read_csv_auto(?)
    GROUP BY ProductCD
    ORDER BY fraud_rate_pct DESC
""", [csv_path]).df()

display(product_risk)

# 2. Build lightweight SVG bar chart
max_rate = product_risk["fraud_rate_pct"].max()
height = 90 + len(product_risk) * 52

svg = f"""
<svg xmlns="http://www.w3.org/2000/svg"
     width="850" height="{height}">
  <text x="20" y="32" font-size="20" font-weight="bold">
    Fraud Rate by Product Category
  </text>
"""

for i, row in product_risk.iterrows():
    y = 60 + i * 52
    width = 450 * row["fraud_rate_pct"] / max_rate

    svg += f"""
    <text x="25" y="{y + 19}" font-size="15">
        {row['ProductCD']}
    </text>

    <rect x="100" y="{y}" width="450" height="28"
          fill="#e5e7eb"/>

    <rect x="100" y="{y}" width="{width:.1f}" height="28"
          fill="#287D91"/>

    <text x="565" y="{y + 19}" font-size="14">
        {row['fraud_rate_pct']:.2f}%
        ({row['fraud_cases']:,} / {row['total_transactions']:,})
    </text>
    """

svg += "</svg>"

display(SVG(data=svg))


In [ ]:

import numpy as np
from scipy.stats import chi2_contingency

# Create contingency table
fraud = product_risk["fraud_cases"].to_numpy()

legitimate = (
    product_risk["total_transactions"]
    - product_risk["fraud_cases"]
).to_numpy()

observed = np.column_stack([legitimate, fraud])

# Chi-Square Test of Independence
chi2, p_value, dof, expected = chi2_contingency(observed)

# Effect size: Cramer's V
n = observed.sum()

cramers_v = np.sqrt(
    chi2 / (n * min(observed.shape[0] - 1,
                    observed.shape[1] - 1))
)

print("Chi-Square Test: ProductCD vs isFraud")
print("-" * 42)

print(f"Chi-Square Statistic : {chi2:.4f}")
print(f"Degrees of Freedom  : {dof}")
print(f"P-value             : {p_value:.6e}")
print(f"Cramer's V          : {cramers_v:.4f}")
print(f"Minimum Expected Count: {expected.min():.2f}")

if p_value < 0.05:
    print("\nDecision: Reject H0")
    print("Product category and fraud status are associated.")
else:
    print("\nDecision: Fail to reject H0")
    print("Insufficient evidence of an association.")


In [ ]:

import duckdb
from IPython.display import SVG, display

# Dataset path
csv_path = str(TRAIN_CSV)

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

# Fraud statistics by card type
card_risk = con.execute("""
    SELECT
        COALESCE(card4, 'Missing') AS card_type,
        COUNT(*) AS total_transactions,
        SUM(isFraud)::BIGINT AS fraud_cases,
        ROUND(100.0 * AVG(isFraud), 2) AS fraud_rate_pct
    FROM read_csv_auto(?)
    GROUP BY 1
    ORDER BY fraud_rate_pct DESC
""", [csv_path]).df()

display(card_risk)

# Lightweight SVG chart
max_rate = card_risk["fraud_rate_pct"].max()
height = 85 + len(card_risk) * 50

svg = f"""
<svg xmlns="http://www.w3.org/2000/svg"
     width="850" height="{height}">
<text x="20" y="30" font-size="20" font-weight="bold">
    Fraud Rate by Payment Card Type
</text>
"""

for i, row in card_risk.iterrows():
    y = 55 + i * 50
    bar_width = 400 * row["fraud_rate_pct"] / max_rate

    svg += f"""
    <text x="20" y="{y+19}" font-size="14">
        {row['card_type']}
    </text>

    <rect x="155" y="{y}" width="400" height="26"
          fill="#e5e7eb"/>

    <rect x="155" y="{y}" width="{bar_width:.1f}"
          height="26" fill="#287D91"/>

    <text x="570" y="{y+19}" font-size="14">
        {row['fraud_rate_pct']:.2f}%
        ({row['fraud_cases']:,} / {row['total_transactions']:,})
    </text>
    """

svg += "</svg>"

display(SVG(data=svg))


In [ ]:

import numpy as np
from scipy.stats import chi2_contingency

# Create contingency table using existing card_risk
fraud = card_risk["fraud_cases"].to_numpy()

legitimate = (
    card_risk["total_transactions"]
    - card_risk["fraud_cases"]
).to_numpy()

observed = np.column_stack([legitimate, fraud])

# Chi-Square Test of Independence
chi2_stat, p_value, dof, expected = chi2_contingency(
    observed
)

# Calculate Cramer's V
n = observed.sum()

cramers_v = np.sqrt(
    chi2_stat / (n * min(
        observed.shape[0] - 1,
        observed.shape[1] - 1
    ))
)

print("CHI-SQUARE TEST: CARD TYPE VS FRAUD")
print("-" * 42)

print(f"Sample Size          : {n:,}")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom  : {dof}")
print(f"P-value             : {p_value:.6e}")
print(f"Cramer's V          : {cramers_v:.4f}")
print(f"Minimum Expected    : {expected.min():.2f}")

print("\nHypothesis Test Result:")

if p_value < 0.05:
    print("Reject H0: Significant association.")
else:
    print("Fail to reject H0: No significant association detected.")

print("\nExpected Frequency Check:")

if expected.min() >= 5:
    print("All expected frequencies >= 5.")
else:
    print("Warning: Some expected frequencies < 5.")


In [ ]:

from IPython.display import SVG, display
from html import escape

# Fraud statistics by card6
card6_risk = con.execute("""
    SELECT
        COALESCE(card6, 'Missing') AS card_category,
        COUNT(*) AS total_transactions,
        SUM(isFraud)::BIGINT AS fraud_cases,
        ROUND(100.0 * AVG(isFraud), 2) AS fraud_rate_pct
    FROM read_csv_auto(?)
    GROUP BY 1
    ORDER BY fraud_rate_pct DESC
""", [csv_path]).df()

display(card6_risk)

# Create lightweight SVG chart
max_rate = card6_risk["fraud_rate_pct"].max()
height = 85 + len(card6_risk) * 50

svg = f"""
<svg xmlns="http://www.w3.org/2000/svg"
     width="850" height="{height}">
<text x="20" y="30" font-size="20" font-weight="bold">
    Fraud Rate by Card Category (card6)
</text>
"""

for i, row in card6_risk.iterrows():
    y = 55 + i * 50
    bar_width = 400 * row["fraud_rate_pct"] / max_rate

    svg += f"""
    <text x="20" y="{y + 19}" font-size="14">
        {escape(str(row['card_category']))}
    </text>

    <rect x="155" y="{y}" width="400" height="26"
          fill="#e5e7eb"/>

    <rect x="155" y="{y}" width="{bar_width:.1f}"
          height="26" fill="#287D91"/>

    <text x="570" y="{y + 19}" font-size="14">
        {row['fraud_rate_pct']:.2f}%
        ({row['fraud_cases']:,} / {row['total_transactions']:,})
    </text>
    """

svg += "</svg>"

display(SVG(data=svg))


In [ ]:

import numpy as np
from scipy.stats import chi2_contingency

# Fraud and legitimate transaction counts
fraud = card6_risk["fraud_cases"].to_numpy()

legitimate = (
    card6_risk["total_transactions"]
    - card6_risk["fraud_cases"]
).to_numpy()

# Construct contingency table
observed = np.column_stack([legitimate, fraud])

# Chi-Square Test of Independence
chi2_stat, p_value, dof, expected = chi2_contingency(
    observed
)

# Calculate Cramer's V
n = observed.sum()

cramers_v = np.sqrt(
    chi2_stat / (
        n * min(observed.shape[0] - 1,
                observed.shape[1] - 1)
    )
)

print("CHI-SQUARE TEST: card6 vs isFraud")
print("-" * 42)

print(f"Sample Size          : {n:,}")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom  : {dof}")
print(f"P-value             : {p_value:.6e}")
print(f"Cramer's V          : {cramers_v:.4f}")
print(f"Minimum Expected    : {expected.min():.2f}")

print("\nHypothesis Test Result:")

if p_value < 0.05:
    print("Reject H0: Statistically significant association.")
else:
    print("Fail to reject H0: Insufficient evidence of association.")

print("\nExpected Frequency Check:")

if expected.min() >= 5:
    print("All expected frequencies >= 5.")
else:
    print("Warning: Some expected frequencies < 5.")
    print("Chi-square approximation needs further examination.")


In [ ]:

import duckdb
from pathlib import Path
from IPython.display import display

# Dataset paths
data_dir = DATA_DIR

csv_path = str(data_dir / "train_transaction.csv")
identity_path = str(data_dir / "train_identity.csv")

# DuckDB connection
con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

# SQL-safe paths
transaction_sql = csv_path.replace("'", "''")
identity_sql = identity_path.replace("'", "''")

# Create a virtual merged dataset
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW fraud_merged AS
    SELECT
        t.*,
        i.* EXCLUDE (TransactionID),
        i.TransactionID AS identity_match_id
    FROM read_csv_auto('{transaction_sql}') AS t
    LEFT JOIN read_csv_auto('{identity_sql}') AS i
        USING (TransactionID)
""")

# Verify the merge
merge_summary = con.execute("""
    SELECT
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT TransactionID) AS unique_transactions,
        COUNT(identity_match_id) AS matched_identity,
        COUNT(*) - COUNT(identity_match_id) AS missing_identity,
        SUM(isFraud)::BIGINT AS fraud_cases
    FROM fraud_merged
""").df()

display(merge_summary)


In [ ]:

import pandas as pd
from IPython.display import display

# 1. Inspect merged dataset columns and data types
schema = con.execute("DESCRIBE fraud_merged").df()

print("Total columns:", len(schema))
print("\nData type distribution:")
display(schema["column_type"].value_counts())

# 2. Select important features for missing-value analysis
features = [
    "TransactionAmt", "ProductCD",
    "card4", "card6", "addr1",
    "dist1", "dist2", "P_emaildomain",
    "R_emaildomain", "D7",
    "id_01", "id_02",
    "DeviceType", "DeviceInfo"
]

# Include only columns available in the merged dataset
available = set(schema["column_name"])
features = [col for col in features if col in available]

# 3. Calculate missing percentages using DuckDB
expressions = [
    f'ROUND(100.0 * AVG(CAST("{col}" IS NULL AS INTEGER)), 2) AS "{col}"'
    for col in features
]

query = "SELECT " + ", ".join(expressions) + " FROM fraud_merged"

missing_result = con.execute(query).df()

# 4. Display missing-value summary
missing_summary = pd.DataFrame({
    "feature": features,
    "missing_pct": missing_result.iloc[0].values
}).sort_values("missing_pct", ascending=False)

display(missing_summary)


In [ ]:

# STEP 27: Chronological Train-Validation Split

# Calculate time cutoff if not already available
if "cutoff_dt" not in globals():
    cutoff_dt = con.execute("""
        SELECT quantile_cont(TransactionDT, 0.8)
        FROM read_csv_auto(?)
    """, [csv_path]).fetchone()[0]

cutoff_dt = float(cutoff_dt)

# Create training view (earlier transactions)
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW fraud_train AS
    SELECT *
    FROM fraud_merged
    WHERE TransactionDT <= {cutoff_dt}
""")

# Create validation view (later transactions)
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW fraud_valid AS
    SELECT *
    FROM fraud_merged
    WHERE TransactionDT > {cutoff_dt}
""")

# Verify both datasets
split_summary = con.execute("""
    SELECT
        'Training' AS dataset,
        COUNT(*) AS transactions,
        SUM(isFraud)::BIGINT AS fraud_cases,
        ROUND(100.0 * AVG(isFraud), 2) AS fraud_rate_pct
    FROM fraud_train

    UNION ALL

    SELECT
        'Validation',
        COUNT(*),
        SUM(isFraud)::BIGINT,
        ROUND(100.0 * AVG(isFraud), 2)
    FROM fraud_valid
""").df()

display(split_summary)

print("Total transactions:", split_summary["transactions"].sum())
print("Time cutoff:", cutoff_dt)


In [ ]:

import pandas as pd
from IPython.display import display

# STEP 28: Select features for the first ML model

candidate_features = [
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4",
    "card5", "card6",
    "addr1", "addr2",
    "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C4",
    "D1", "D2", "D7",
    "M1", "M4",
    "id_01", "id_02",
    "DeviceType", "DeviceInfo"
]

# Card identifiers and other codes are categorical
categorical_features = [
    "ProductCD",
    "card1", "card2", "card3", "card4",
    "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "M1", "M4",
    "DeviceType", "DeviceInfo"
]

# Read column metadata (not the full dataset)
schema = con.execute("DESCRIBE fraud_train").df()

data_types = dict(zip(
    schema["column_name"],
    schema["column_type"]
))

features = [
    col for col in candidate_features
    if col in data_types
]

# Calculate missing percentages from TRAINING data only
missing_sql = ", ".join(
    f'ROUND(100.0 * AVG(CASE WHEN "{col}" IS NULL '
    f'THEN 1 ELSE 0 END), 2) AS "{col}"'
    for col in features
)

missing_values = con.execute(
    f"SELECT {missing_sql} FROM fraud_train"
).df().iloc[0]

# Create feature summary
feature_report = pd.DataFrame({
    "feature": features,
    "data_type": [data_types[col] for col in features],
    "feature_type": [
        "Categorical" if col in categorical_features
        else "Numerical"
        for col in features
    ],
    "missing_pct": [
        missing_values[col] for col in features
    ]
})

feature_report = feature_report.sort_values(
    "missing_pct", ascending=False
).reset_index(drop=True)

print("Selected features:", len(features))
print("\nFeature type distribution:")

display(feature_report["feature_type"].value_counts())

print("\nTraining Data Missing-Value Report:")
display(feature_report)


In [ ]:

# STEP 29: Prepare ML Training and Validation Data

import pandas as pd

# Use the 26 features selected in Step 28
feature_cols = features.copy()

# Load only selected columns to save memory
selected_cols = ", ".join(
    f'"{col}"' for col in feature_cols
)

train_df = con.execute(f"""
    SELECT {selected_cols}, isFraud
    FROM fraud_train
""").df()

valid_df = con.execute(f"""
    SELECT {selected_cols}, isFraud
    FROM fraud_valid
""").df()

# Separate features (X) and target (y)
y_train = train_df.pop("isFraud").astype("int8")
y_valid = valid_df.pop("isFraud").astype("int8")

X_train = train_df
X_valid = valid_df

# Convert categorical features
cat_cols = [
    col for col in categorical_features
    if col in feature_cols
]

for col in cat_cols:
    X_train[col] = X_train[col].astype("string").astype("category")

    # Validation uses training categories only
    X_valid[col] = pd.Categorical(
        X_valid[col].astype("string"),
        categories=X_train[col].cat.categories
    )

# Convert numerical features to memory-efficient float32
num_cols = [
    col for col in feature_cols
    if col not in cat_cols
]

for col in num_cols:
    X_train[col] = X_train[col].astype("float32")
    X_valid[col] = X_valid[col].astype("float32")

# Verify results
print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("\nFraud distribution:")
print("Training fraud rate:", round(y_train.mean() * 100, 2), "%")
print("Validation fraud rate:", round(y_valid.mean() * 100, 2), "%")

print("\nCategorical features:", len(cat_cols))
print("Numerical features:", len(num_cols))

print("\nMemory usage:")
print("Training:", round(X_train.memory_usage(deep=True).sum() / 1024**2, 2), "MB")
print("Validation:", round(X_valid.memory_usage(deep=True).sum() / 1024**2, 2), "MB")


In [ ]:

# STEP 30: LightGBM Baseline Model

import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Create baseline model
model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

# 2. Train using earlier transactions only
model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols
)

# 3. Predict fraud probabilities
y_prob = model.predict_proba(X_valid)[:, 1]

# 4. Convert probabilities to fraud predictions
threshold = 0.20
y_pred = (y_prob >= threshold).astype(int)

# 5. Evaluate model
print("LIGHTGBM BASELINE RESULTS")
print("-" * 35)

print("ROC-AUC:",
      round(roc_auc_score(y_valid, y_prob), 4))

print("PR-AUC:",
      round(average_precision_score(y_valid, y_prob), 4))

print("Precision:",
      round(precision_score(y_valid, y_pred, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_valid, y_pred, zero_division=0), 4))

print("F1-score:",
      round(f1_score(y_valid, y_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, y_pred))


In [ ]:

# STEP 31: Compare Fraud Detection Thresholds

from sklearn.metrics import precision_score, recall_score, f1_score

print(f"{'Threshold':<12}{'Precision':<12}{'Recall':<12}{'F1-score':<12}")
print("-" * 48)

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    predictions = (y_prob >= threshold).astype(int)

    precision = precision_score(y_valid, predictions, zero_division=0)
    recall = recall_score(y_valid, predictions, zero_division=0)
    f1 = f1_score(y_valid, predictions, zero_division=0)

    print(f"{threshold:<12.2f}{precision:<12.4f}{recall:<12.4f}{f1:<12.4f}")


In [ ]:

# STEP 32: LightGBM with Class Weighting

import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Calculate class imbalance
legitimate_count = (y_train == 0).sum()
fraud_count = (y_train == 1).sum()

scale_weight = legitimate_count / fraud_count

print("Legitimate transactions:", legitimate_count)
print("Fraud transactions:", fraud_count)
print("Class weight:", round(scale_weight, 2))

# 2. Train weighted LightGBM
weighted_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

weighted_model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols
)

# 3. Predict probabilities
weighted_prob = weighted_model.predict_proba(X_valid)[:, 1]

# 4. Evaluate at the same threshold
threshold = 0.20
weighted_pred = (weighted_prob >= threshold).astype(int)

print("\nWEIGHTED LIGHTGBM RESULTS")
print("-" * 35)

print("ROC-AUC:",
      round(roc_auc_score(y_valid, weighted_prob), 4))

print("PR-AUC:",
      round(average_precision_score(y_valid, weighted_prob), 4))

print("Precision:",
      round(precision_score(y_valid, weighted_pred, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_valid, weighted_pred, zero_division=0), 4))

print("F1-score:",
      round(f1_score(y_valid, weighted_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, weighted_pred))


In [ ]:

# STEP 33: Compare Both Models at Different Thresholds

import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

results = []

models = {
    "Baseline": y_prob,
    "Weighted": weighted_prob
}

for model_name, probabilities in models.items():

    for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_valid, predictions, labels=[0, 1]
        ).ravel()

        results.append({
            "Model": model_name,
            "Threshold": threshold,
            "ROC-AUC": roc_auc_score(y_valid, probabilities),
            "PR-AUC": average_precision_score(y_valid, probabilities),
            "Precision": precision_score(
                y_valid, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_valid, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_valid, predictions, zero_division=0
            ),
            "False Positives": fp,
            "False Negatives": fn
        })

comparison_df = pd.DataFrame(results)

print("MODEL COMPARISON")
display(comparison_df.round(4))

print("\nBEST F1-SCORE FOR EACH MODEL")

best_results = comparison_df.loc[
    comparison_df.groupby("Model")["F1"].idxmax()
]

display(best_results.round(4))


In [ ]:

# STEP 34: Feature Importance Analysis

import pandas as pd

# Compare feature importance for both models
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Baseline_Gain": model.booster_.feature_importance(
        importance_type="gain"
    ),
    "Weighted_Gain": weighted_model.booster_.feature_importance(
        importance_type="gain"
    )
})

# Convert gain into percentage of total gain
importance_df["Baseline_%"] = (
    importance_df["Baseline_Gain"]
    / importance_df["Baseline_Gain"].sum()
    * 100
)

importance_df["Weighted_%"] = (
    importance_df["Weighted_Gain"]
    / importance_df["Weighted_Gain"].sum()
    * 100
)

# Sort by baseline importance
importance_df = importance_df.sort_values(
    "Baseline_%",
    ascending=False
).reset_index(drop=True)

print("TOP 15 MOST IMPORTANT FEATURES")
display(
    importance_df[
        ["Feature", "Baseline_%", "Weighted_%"]
    ].head(15).round(2)
)


In [ ]:

# STEP 35: Train LightGBM Using Top 20 Features

import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Select top 20 features from Step 34
top_features = importance_df.head(20)["Feature"].tolist()

print("Selected features:", len(top_features))
print(top_features)

# 2. Create reduced datasets
X_train_top = X_train[top_features]
X_valid_top = X_valid[top_features]

# 3. Keep categorical features present in top 20
top_cat_cols = [
    col for col in cat_cols
    if col in top_features
]

# 4. Train model with same baseline settings
top_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

top_model.fit(
    X_train_top,
    y_train,
    categorical_feature=top_cat_cols
)

# 5. Predict validation probabilities
top_prob = top_model.predict_proba(X_valid_top)[:, 1]

# 6. Evaluate at threshold 0.20
threshold = 0.20
top_pred = (top_prob >= threshold).astype(int)

print("\nTOP 20 FEATURES MODEL")
print("-" * 35)

print("ROC-AUC:",
      round(roc_auc_score(y_valid, top_prob), 4))

print("PR-AUC:",
      round(average_precision_score(y_valid, top_prob), 4))

print("Precision:",
      round(precision_score(y_valid, top_pred, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_valid, top_pred, zero_division=0), 4))

print("F1-score:",
      round(f1_score(y_valid, top_pred, zero_division=0), 4))


In [ ]:

# STEP 36: Final Model Comparison

import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Models and their existing predictions
model_predictions = {
    "Baseline (26)": y_prob,
    "Weighted (26)": weighted_prob,
    "Top 20": top_prob
}

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]
results = []

for name, probabilities in model_predictions.items():
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_valid, predictions, labels=[0, 1]
        ).ravel()

        results.append({
            "Model": name,
            "Threshold": threshold,
            "ROC-AUC": roc_auc_score(y_valid, probabilities),
            "PR-AUC": average_precision_score(y_valid, probabilities),
            "Precision": precision_score(
                y_valid, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_valid, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_valid, predictions, zero_division=0
            ),
            "FP": fp,
            "FN": fn
        })

comparison = pd.DataFrame(results)

# Best tested threshold for each model
best_per_model = comparison.loc[
    comparison.groupby("Model")["F1"].idxmax()
].sort_values("F1", ascending=False)

print("BEST CONFIGURATION FOR EACH MODEL")
display(best_per_model.round(4))

# Select highest validation F1-score
winner = best_per_model.iloc[0]

best_model_name = winner["Model"]
best_threshold = float(winner["Threshold"])

best_model = {
    "Baseline (26)": model,
    "Weighted (26)": weighted_model,
    "Top 20": top_model
}[best_model_name]

best_features = (
    top_features
    if best_model_name == "Top 20"
    else list(X_train.columns)
)

print("\nSELECTED MODEL:", best_model_name)
print("SELECTED THRESHOLD:", best_threshold)
print("FEATURE COUNT:", len(best_features))
print("VALIDATION F1:", round(winner["F1"], 4))
print("VALIDATION PR-AUC:", round(winner["PR-AUC"], 4))


In [ ]:

# STEP 37: Prepare Full Training Data

import pandas as pd

X_full = pd.concat(
    [X_train[best_features], X_valid[best_features]],
    ignore_index=True
)

y_full = pd.concat(
    [y_train, y_valid],
    ignore_index=True
)

print("Full training shape:", X_full.shape)
print("Labels shape:", y_full.shape)
print("Fraud rate:", round(y_full.mean() * 100, 2), "%")

assert len(X_full) == len(y_full)
assert len(X_full) == 590540
assert X_full.columns.tolist() == best_features


In [ ]:

# STEP 38: Train Final LightGBM Model

from sklearn.base import clone

# Select the winning model configuration
selected_model = {
    "Baseline (26)": model,
    "Weighted (26)": weighted_model,
    "Top 20": top_model
}[best_model_name]

final_model = clone(selected_model)

# Recalculate fraud weight using all labelled data
if best_model_name == "Weighted (26)":
    final_weight = (y_full == 0).sum() / (y_full == 1).sum()
    final_model.set_params(scale_pos_weight=final_weight)

final_cat_cols = [
    col for col in cat_cols
    if col in best_features
]

final_model.fit(
    X_full,
    y_full,
    categorical_feature=final_cat_cols
)

print("Final model trained successfully")
print("Model:", best_model_name)
print("Features:", len(best_features))
print("Threshold:", best_threshold)
print("Training rows:", len(X_full))


In [ ]:

# STEP 39: Load Kaggle Test Data

from pathlib import Path

data_dir = DATA_DIR

test_transaction_path = data_dir / "test_transaction.csv"
test_identity_path = data_dir / "test_identity.csv"

# Load only required transaction columns
transaction_columns = pd.read_csv(
    test_transaction_path,
    nrows=0
).columns.tolist()

test_transaction_features = [
    col for col in best_features
    if col in transaction_columns
]

test_transaction = pd.read_csv(
    test_transaction_path,
    usecols=["TransactionID"] + test_transaction_features
)

print("Test transaction shape:", test_transaction.shape)
print("Test transaction rows:", len(test_transaction))


In [ ]:

# STEP 40: Merge Kaggle Test Data

# Read identity header to select only necessary columns
identity_columns = pd.read_csv(
    test_identity_path,
    nrows=0
).columns.tolist()

identity_feature_columns = [
    col for col in identity_columns
    if col.replace("-", "_") in best_features
]

test_identity = pd.read_csv(
    test_identity_path,
    usecols=["TransactionID"] + identity_feature_columns
)

# Standardise identity feature names
test_identity = test_identity.rename(
    columns=lambda col: col.replace("-", "_")
    if col != "TransactionID" else col
)

# One identity record per transaction
assert test_identity["TransactionID"].is_unique

test_merged = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left",
    validate="one_to_one"
)

assert len(test_merged) == len(test_transaction)

print("Merged test shape:", test_merged.shape)
print("Rows preserved:", len(test_merged))


In [ ]:

# STEP 41: Prepare Test Features

missing_features = [
    col for col in best_features
    if col not in test_merged.columns
]

if missing_features:
    raise ValueError(
        f"Missing test features: {missing_features}"
    )

X_test = test_merged[best_features].copy()

# Match categorical types and training categories
for col in final_cat_cols:
    X_test[col] = pd.Categorical(
        X_test[col].astype("string"),
        categories=X_full[col].cat.categories
    )

# Match numerical types
for col in best_features:
    if col not in final_cat_cols:
        X_test[col] = pd.to_numeric(
            X_test[col],
            errors="coerce"
        ).astype("float32")

print("Test feature shape:", X_test.shape)
print("Feature order matches:",
      X_test.columns.tolist() == best_features)

assert X_test.columns.tolist() == best_features
assert len(X_test) == len(test_merged)


In [ ]:

# STEP 42: Generate Test Fraud Predictions

import numpy as np

test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

# Optional binary decisions for our analysis
test_predictions = (
    test_probabilities >= best_threshold
).astype("int8")

print("Total test transactions:", len(test_probabilities))
print("Predicted fraud cases:", int(test_predictions.sum()))
print("Predicted fraud rate:",
      round(test_predictions.mean() * 100, 2), "%")

print("\nProbability statistics:")
print(pd.Series(test_probabilities).describe())

assert np.isfinite(test_probabilities).all()
assert ((test_probabilities >= 0) &
        (test_probabilities <= 1)).all()


In [ ]:

# STEP 43: Generate Kaggle Submission File

submission = pd.DataFrame({
    "TransactionID": test_merged["TransactionID"],
    "isFraud": test_probabilities
})

submission_path = OUTPUT_DIR / "submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Submission saved:", submission_path)
print("Submission shape:", submission.shape)

print("\nFirst 5 predictions:")
display(submission.head())

# Final checks
assert submission.columns.tolist() == [
    "TransactionID", "isFraud"
]
assert submission["TransactionID"].is_unique
assert submission["isFraud"].between(0, 1).all()

print("\nSubmission validation passed.")


In [ ]:

# STEP 44: Save Final Model and Preprocessing Metadata

import joblib
from pathlib import Path

model_dir = OUTPUT_DIR / "saved_model"
model_dir.mkdir(parents=True, exist_ok=True)

model_package = {
    "model": final_model,
    "features": best_features,
    "categorical_features": final_cat_cols,
    "category_mappings": {
        col: X_full[col].cat.categories.tolist()
        for col in final_cat_cols
    },
    "threshold": best_threshold,
    "model_name": best_model_name
}

model_path = model_dir / "fraud_detection_model.joblib"

joblib.dump(model_package, model_path)

# Verify saved model can be loaded
loaded_package = joblib.load(model_path)

print("Model saved successfully!")
print("Location:", model_path)
print("Model:", loaded_package["model_name"])
print("Features:", len(loaded_package["features"]))
print("Threshold:", loaded_package["threshold"])
print("File size:",
      round(model_path.stat().st_size / 1024**2, 2),
      "MB")


In [ ]:

# STEP 45: Verify Saved Model Predictions

import joblib
import numpy as np

# 1. Load saved model package
saved_package = joblib.load(model_path)

loaded_model = saved_package["model"]
saved_features = saved_package["features"]
saved_threshold = saved_package["threshold"]

# 2. Verify feature order
assert X_test.columns.tolist() == saved_features

# 3. Predict using loaded model
loaded_probabilities = loaded_model.predict_proba(
    X_test
)[:, 1]

# 4. Compare with original predictions
same_predictions = np.allclose(
    test_probabilities,
    loaded_probabilities,
    rtol=1e-10,
    atol=1e-12
)

max_difference = np.max(
    np.abs(test_probabilities - loaded_probabilities)
)

print("MODEL VERIFICATION")
print("-" * 35)

print("Saved model:", saved_package["model_name"])
print("Features:", len(saved_features))
print("Threshold:", saved_threshold)

print("\nPredictions match:", same_predictions)
print("Maximum difference:", max_difference)

assert same_predictions, "Saved model predictions differ!"

print("\nVerification successful!")


In [ ]:

# STEP 46: Selected Model Evaluation

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
import pandas as pd

validation_prob = {
    "Baseline (26)": y_prob,
    "Weighted (26)": weighted_prob,
    "Top 20": top_prob
}[best_model_name]

validation_pred = (
    validation_prob >= best_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_valid, validation_pred, labels=[0, 1]
).ravel()

metrics = {
    "Model": best_model_name,
    "Threshold": float(best_threshold),
    "ROC-AUC": roc_auc_score(y_valid, validation_prob),
    "PR-AUC": average_precision_score(
        y_valid, validation_prob
    ),
    "Precision": precision_score(
        y_valid, validation_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_valid, validation_pred, zero_division=0
    ),
    "F1": f1_score(
        y_valid, validation_pred, zero_division=0
    ),
    "True Positives": int(tp),
    "False Positives": int(fp),
    "False Negatives": int(fn),
    "True Negatives": int(tn)
}

metrics_df = pd.DataFrame([metrics])
display(metrics_df.T)

print("Fraud detected:", tp)
print("Fraud missed:", fn)
print("False alarms:", fp)


In [ ]:

# ============================================================
# IEEE-CIS FRAUD DETECTION — STEPS 47 TO 53
# ============================================================

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
    confusion_matrix
)

report_dir = OUTPUT_DIR / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# STEP 47: SAVE MODEL COMPARISON AND VALIDATION METRICS
# ------------------------------------------------------------

comparison.to_csv(
    report_dir / "model_comparison.csv",
    index=False
)

metrics_df.to_csv(
    report_dir / "validation_metrics.csv",
    index=False
)

print("STEP 47: Evaluation CSVs saved")


# ------------------------------------------------------------
# STEP 48: PRECISION-RECALL CURVES
# ------------------------------------------------------------

probability_sets = {
    "Baseline": y_prob,
    "Weighted": weighted_prob,
    "Top 20": top_prob
}

fig = go.Figure()

for name, probabilities in probability_sets.items():
    precision, recall, _ = precision_recall_curve(
        y_valid, probabilities
    )

    fig.add_trace(go.Scatter(
        x=recall,
        y=precision,
        mode="lines",
        name=name
    ))

fig.update_layout(
    title="Precision-Recall Curve Comparison",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_white",
    width=850,
    height=500
)

fig.write_html(
    str(report_dir / "precision_recall_curves.html")
)

fig.show()
print("STEP 48: Precision-recall chart saved")


# ------------------------------------------------------------
# STEP 49: ROC CURVES
# ------------------------------------------------------------

fig = go.Figure()

for name, probabilities in probability_sets.items():
    fpr, tpr, _ = roc_curve(
        y_valid, probabilities
    )

    auc = roc_auc_score(
        y_valid, probabilities
    )

    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"{name} (AUC={auc:.3f})"
    ))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Random classifier",
    line=dict(dash="dash", color="gray")
))

fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=850,
    height=500
)

fig.write_html(
    str(report_dir / "roc_curves.html")
)

fig.show()
print("STEP 49: ROC chart saved")


# ------------------------------------------------------------
# STEP 50: CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_valid,
    validation_pred,
    labels=[0, 1]
)

fig = px.imshow(
    cm,
    x=["Predicted Legitimate", "Predicted Fraud"],
    y=["Actual Legitimate", "Actual Fraud"],
    text_auto=True,
    color_continuous_scale="Blues",
    title=f"Confusion Matrix — {best_model_name}"
)

fig.update_layout(
    width=750,
    height=550
)

fig.write_html(
    str(report_dir / "confusion_matrix.html")
)

fig.show()
print("STEP 50: Confusion matrix saved")


# ------------------------------------------------------------
# STEP 51: FINAL FEATURE IMPORTANCE
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "Feature": best_features,
    "Gain": final_model.booster_.feature_importance(
        importance_type="gain"
    )
})

total_gain = feature_importance["Gain"].sum()

feature_importance["Importance_%"] = (
    feature_importance["Gain"] / total_gain * 100
    if total_gain > 0 else 0.0
)

feature_importance = feature_importance.sort_values(
    "Importance_",
    ascending=False
) if "Importance_" in feature_importance.columns else (
    feature_importance.sort_values(
        "Importance_%",
        ascending=False
    )
)

feature_importance.to_csv(
    report_dir / "feature_importance.csv",
    index=False
)

fig = px.bar(
    feature_importance.head(15).sort_values("Importance_%"),
    x="Importance_%",
    y="Feature",
    orientation="h",
    title="Top 15 Fraud Detection Features"
)

fig.update_layout(
    template="plotly_white",
    width=850,
    height=550
)

fig.write_html(
    str(report_dir / "feature_importance.html")
)

fig.show()
print("STEP 51: Feature importance saved")

print("Evaluation tables and charts saved to:", report_dir)
print("Static project documentation is maintained in README.md and PROJECT_FINDINGS.md.")
